In [48]:
import os
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-3.1-pro-preview'
from google import genai
from google.genai import types
# Only run this block for Gemini Developer API
client = genai.Client(api_key=GEMINI_API_KEY)

In [5]:
from datetime import datetime
data_corrente = datetime.now().strftime("%A, %d %B %Y")
SYSTEM_MESSAGE = f"""
Ti muovi in un ciclo di Thought (Pensiero), Action (Azione), PAUSE (Pausa), Observation (Osservazione).
Al termine del ciclo, restituisci una Answer (Risposta).

Usa sempre Thought come primo step per descrivere i tuoi pensieri riguardo alla domanda che ti è stata posta.
Usa Action per eseguire una delle azioni a tua disposizione - poi restituisci PAUSE.
Observation sarà il risultato dell'esecuzione di tali azioni.

Le tue azioni disponibili sono:

wikipedia:
e.g. wikipedia: Isaac Newton
Cerca su Wikipedia e restituisce un riassunto della pagina o dei risultati rilevanti.


Esempio di sessione:

Question: Quando è nato Albert Einstein?
Thought: Devo cercare la data di nascita di Albert Einstein su Wikipedia.
Action: wikipedia: Albert Einstein
PAUSE

Verrai richiamato con questo:

Observation: Albert Einstein (Ulma, 14 marzo 1879 – Princeton, 18 aprile 1955) è stato un fisico tedesco naturalizzato svizzero e statunitense.

Quindi restituirai:

Answer: Albert Einstein è nato il 14 marzo 1879.

La data corrente è {data_corrente}. 
Rispondi alle domande sapendo che questo è il riferimento temporale odierno.

"""

In [2]:
class Agent:
    def __init__(self, system="", tools = dict()):
        self.system = system
        self.messages = []
        self.tools = tools

    def __call__(self, message):
        self.messages.append(
        types.Content(
            role='user',
            parts=[
        types.Part.from_text(text=message),
    ]
        ))
        result = self.execute()
        self.messages.append(types.Content(
            role='model',
            parts=[
        types.Part.from_text(text=result),
    ]
        ))
        return result

    def execute(self):
        res = client.models.generate_content(
            model=GEMINI_MODEL,
            
            contents=self.messages,
            config=types.GenerateContentConfig(
                system_instruction=self.system,
                temperature=0.3
            )
        )
        return res.text

In [3]:
import wikipediaapi

def wikipedia(q):
    """
    Cerca su Wikipedia e restituisce un riassunto della pagina.
    """
    # È necessario specificare un user_agent appropriato secondo le policy di Wiki
    wiki_wiki = wikipediaapi.Wikipedia(
        user_agent='TestAgenteAI per corso (andreacds3@gmail.com)',
        language='it',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    page = wiki_wiki.page(q)

    if page.exists():
        return page.summary
    else:
        # In caso di pagina non trovata, proviamo a suggerire termini simili
        return f"Errore: La pagina per '{q}' non è stata trovata."    

In [50]:
bot = Agent(SYSTEM_MESSAGE)

In [49]:
message = 'Chi ha vinto il campionato di Calcio nel 2025 in Italia e di quella squadra quale calciatore guadagnava di più?'

In [51]:
known_actions = {'wikipedia': wikipedia}

In [52]:
import re
action_re = re.compile(r'^Action: (\w+): (.*)$')   # python regular expression to selection action
def query(question,agent,max_turns=5):
    i = 0
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return
    

In [53]:
query(message, bot)

Thought: Devo cercare quale squadra ha vinto il campionato di Serie A nella stagione 2024-2025.
Action: wikipedia: Serie A 2024-2025
PAUSE
 -- running wikipedia Serie A 2024-2025
Observation: La Serie A 2024-2025 è stata la 123ª edizione della massima serie del campionato italiano di calcio (la 93ª a girone unico), disputata tra il 17 agosto 2024 e il 25 maggio 2025 e conclusa con la vittoria del Napoli, al suo quarto titolo.
Il capocannoniere della stagione è stato Mateo Retegui (Atalanta) con 25 reti.
Thought: Il Napoli ha vinto il campionato di Serie A 2024-2025. Ora devo scoprire quale calciatore della rosa del Napoli aveva lo stipendio più alto in quella stagione. Cerco la pagina della stagione 2024-2025 del Napoli.
Action: wikipedia: Società Sportiva Calcio Napoli 2024-2025
PAUSE
 -- running wikipedia Società Sportiva Calcio Napoli 2024-2025
Observation: Questa voce raccoglie le informazioni riguardanti la Società Sportiva Calcio Napoli nelle competizioni ufficiali della stagione

AttributeError: 'NoneType' object has no attribute 'split'

In [36]:
SYSTEM_MESSAGE_2 = f"""
Ti muovi in un ciclo di Thought (Pensiero), Action (Azione), PAUSE (Pausa), Observation (Osservazione).
Al termine del ciclo, restituisci una Answer (Risposta).

Usa sempre Thought come primo step per descrivere i tuoi pensieri riguardo alla domanda che ti è stata posta.
Usa Action per eseguire una delle azioni a tua disposizione - poi restituisci PAUSE.
Observation sarà il risultato dell'esecuzione di tali azioni.

Le tue azioni disponibili sono:

wikipedia:
e.g. wikipedia: Isaac Newton
Cerca su Wikipedia e restituisce un riassunto della pagina o dei risultati rilevanti.
web_search:
e.g. web_search: Eventi Roma oggi
Cerca su google e restituisce un riassunto delle pagine rilevanti.

Esempio di sessione:

Question: Quando è nato Albert Einstein?
Thought: Devo cercare la data di nascita di Albert Einstein su Wikipedia.
Action: wikipedia: Albert Einstein
PAUSE

Verrai richiamato con questo:

Observation: Albert Einstein (Ulma, 14 marzo 1879 – Princeton, 18 aprile 1955) è stato un fisico tedesco naturalizzato svizzero e statunitense.

Quindi restituirai:

Answer: Albert Einstein è nato il 14 marzo 1879.

La data corrente è {data_corrente}. 
Rispondi alle domande sapendo che questo è il riferimento temporale odierno.

"""

In [37]:
from tavily import TavilyClient
def tavily_search_tool(
    query: str, max_results: int = 5
) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        List[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key)

    try:
        response = client.search(
            query=query, max_results=max_results
        )

        results = []
        for r in response.get("results", []):
            results.append(
                {
                    "title": r.get("title", ""),
                    "content": r.get("content", ""),
                    "url": r.get("url", ""),
                }
            )

        return '\n\n'.join(json.dumps(results))

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [38]:
known_actions['web_search'] = tavily_search_tool

In [42]:
bot_2 = Agent(SYSTEM_MESSAGE_2)

In [45]:
message = 'Eventi Roma'

In [46]:
query(message, bot_2)

Answer: Come modello di linguaggio, non ho accesso a informazioni in tempo reale o a un calendario di eventi attuali. Non posso quindi fornirti un elenco di "Eventi Roma" che si stanno svolgendo oggi o in un periodo specifico.
